# LangGraph

This notebook reproduces the Microsoft Agent Framework examples with LangGraph-backed agents.

## Install dependencies

Uncomment and run this cell once if the packages are not already installed.

In [ ]:
# %pip install -U langchain langgraph langchain-openai python-dotenv

# Install the two tracing packages in a SINGLE command so pip resolves a compatible pair.
# azure-monitor-opentelemetry pins an exact opentelemetry-sdk version, so upgrading the
# langchain instrumentor on its own (with -U) pulls in an SDK that breaks configure_azure_monitor.
# %pip install azure-monitor-opentelemetry opentelemetry-instrumentation-langchain

In [ ]:
import os
from random import randint

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

# Load environment variables from the .env file.
load_dotenv()

# Verify that configuration is present without displaying secret values.
print("Endpoint loaded:", bool(os.getenv("AZURE_OPENAI_ENDPOINT")))
print("API key loaded:", bool(os.getenv("AZURE_OPENAI_API_KEY")))
print("Deployment loaded:", bool(os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")))

## Set logging

In [ ]:
# Send LangChain/LangGraph OpenTelemetry traces to Azure Application Insights.
# Add the connection string to .env as APPLICATIONINSIGHTS_CONNECTION_STRING.
#
# Set ECHO_SPANS_LOCALLY = True to also print spans to this notebook, so you can
# verify instrumentation immediately instead of waiting for portal ingestion.
ECHO_SPANS_LOCALLY = True

from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry.instrumentation.langchain import LangchainInstrumentor

conn = os.getenv("APPLICATIONINSIGHTS_CONNECTION_STRING")
if not conn:
    raise ValueError(
        "APPLICATIONINSIGHTS_CONNECTION_STRING is not set. "
        "Add it to .env from the Application Insights resource overview."
    )

configure_azure_monitor(connection_string=conn)

# instrument() is idempotent-guarded here so re-running this cell does not raise.
instrumentor = LangchainInstrumentor()
if not instrumentor.is_instrumented_by_opentelemetry:
    instrumentor.instrument()

if ECHO_SPANS_LOCALLY:
    from opentelemetry import trace
    from opentelemetry.sdk.trace.export import SimpleSpanProcessor, SpanExporter, SpanExportResult

    class PrintSpanExporter(SpanExporter):
        """Prints a one-line summary for each finished span."""

        def export(self, spans):
            for span in spans:
                duration_ms = (span.end_time - span.start_time) / 1_000_000
                print(f"  [span] {span.name} ({duration_ms:.0f}ms)")
            return SpanExportResult.SUCCESS

        def shutdown(self):
            return None

    provider = trace.get_tracer_provider()
    if not getattr(provider, "_console_echo_added", False):
        provider.add_span_processor(SimpleSpanProcessor(PrintSpanExporter()))
        provider._console_echo_added = True

print("Tracing configured. Traces appear in the portal in about 30-90 seconds.")

## Normal response

In [ ]:
# Initialize the chat model through LangChain's generic initializer.
#
# This resource exposes the OpenAI-compatible Foundry endpoint (.../openai/v1), which is
# why the "openai:" provider plus base_url is used here — the same style as the original
# OpenAIChatClient(base_url=...) example. For a classic Azure OpenAI resource you would
# instead use: init_chat_model(f"azure_openai:{DEPLOYMENT}", azure_endpoint=..., api_version=...)
ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "").rstrip("/")
DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")

client = init_chat_model(
    f"openai:{DEPLOYMENT}",
    base_url=ENDPOINT,
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
)
# print("Model initialized:", type(client).__name__, "->", client.root_client.base_url)

agent = create_agent(
    model=client,
    name="HelloAgent-langgraph",
    system_prompt="You are a friendly assistant. Keep your answers brief.",
)

response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "Say hello and introduce yourself in one sentence."}]}
)
print(response["messages"][-1].content)

## Streaming

In [ ]:
# Stream model tokens as they are generated.
print("Agent (streaming): ", end="", flush=True)
async for message, metadata in agent.astream(
    {"messages": [{"role": "user", "content": "Tell me a one-sentence fun fact."}]},
    stream_mode="messages",
):
    if message.content and metadata.get("langgraph_node") == "model":
        print(message.content, end="", flush=True)
print()

## Tool calling

### Define tool

In [ ]:
@tool
def get_weather(location: str) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    return (
        f"The weather in {location} is {conditions[randint(0, 3)]} "
        f"with a high of {randint(10, 30)}°C."
    )

### Agent with tool

In [ ]:
agent = create_agent(
    model=client,
    name="WeatherAgent-langgraph",
    system_prompt="You are a helpful weather agent. Use the get_weather tool to answer questions.",
    tools=[get_weather],
)

In [ ]:
response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "What's the weather like in Paris today?"}]}
)
print(response["messages"][-1].content)

## Car price agent (parallel lookups + calculator tool)

A single LangGraph-backed agent uses two tools:

- **`get_car_price`** looks up one brand's price. When the model requests multiple calls in one turn, LangGraph's tool node executes them in parallel.
- **`add`** performs deterministic arithmetic so the model does not calculate the total itself.

Flow: parallel `get_car_price` calls → `add` → final answer.

In [ ]:
import asyncio
import time
from datetime import datetime

CAR_PRICES = {"honda": 28000, "toyota": 31000}


@tool
async def get_car_price(brand: str) -> str:
    """Get the price of a single car for the given brand."""
    print(f"[{datetime.now():%H:%M:%S.%f}] get_car_price START (brand={brand})")
    await asyncio.sleep(2)
    price = CAR_PRICES.get(brand.strip().lower())
    result = f"{brand}: ${price}" if price is not None else f"{brand}: unknown brand"
    print(f"[{datetime.now():%H:%M:%S.%f}] get_car_price END   (brand={brand}) -> {result}")
    return result


@tool
def add(numbers: list[float]) -> str:
    """Add a list of numbers and return their total."""
    print(f"[{datetime.now():%H:%M:%S.%f}] add() called with {numbers}")
    return f"The sum of {numbers} is {sum(numbers)}."


car_agent = create_agent(
    model=client,
    name="CarPriceAgent_Parallel-langgraph",
    system_prompt=(
        "You are a car pricing assistant. Use get_car_price to look up each brand's price, "
        "requesting all required brand lookups together so they run in parallel. "
        "For any addition or total, do not calculate it yourself; call the add tool."
    ),
    tools=[get_car_price, add],
)

start = time.perf_counter()
response = await car_agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the total price of 5 Honda car and 2 Toyota cars?",
            }
        ]
    }
)
print(f"\nWall-clock time: {time.perf_counter() - start:.1f}s\n")
print(response["messages"][-1].content)